# 01 — Exploratory Data Analysis
Richter's Predictor: memprediksi `damage_grade` bangunan setelah gempa Gorkha 2015.
Notebook ini hanya eksplorasi; seluruh logika modeling ada di `scripts/`.

In [ ]:
import sys
sys.path.append("..")  # agar package `scripts` bisa di-import dari folder notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts import config
from scripts.preprocess import load_train, load_test, load_metadata, split_xy

train, test, meta = load_train(), load_test(), load_metadata()
print("Train:", train.shape, "| Test:", test.shape)
train.head()

## Deskripsi variabel

In [ ]:
meta

## Distribusi target
Kelas tidak seimbang: Grade 5 paling banyak, Grade 1 paling sedikit. Metrik kompetisi adalah micro-F1 (setara akurasi).

In [ ]:
counts = train[config.TARGET].value_counts().sort_index()
print(counts, "\n")
print(counts.div(len(train)).round(3))

plt.figure(figsize=(7, 4))
sns.countplot(data=train, x=config.TARGET, order=sorted(train[config.TARGET].unique()), color="#3b6ea5")
plt.title("Distribusi Damage Grade")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "target_distribution.png", dpi=120)
plt.show()

## Kualitas data

In [ ]:
print("Missing (train):", train.isna().sum().sum(), "| Missing (test):", test.isna().sum().sum())
print("building_id duplikat:", train[config.ID_COL].duplicated().sum())
print("Kolom train == test (di luar target):",
      train.drop(columns=config.TARGET).columns.tolist() == test.columns.tolist())
summary = pd.DataFrame({"dtype": train.dtypes.astype(str), "n_unique": train.nunique()})
summary

### `age_building == 999`
Nilai 999 adalah kode *umur tidak diketahui*, bukan umur sebenarnya. Pipeline mengubahnya menjadi `NaN` + flag `age_unknown`.

In [ ]:
print("Jumlah 999 -> train:", (train.age_building == 999).sum(), "| test:", (test.age_building == 999).sum())
print(train.loc[train.age_building != 999, "age_building"].describe())

## Hubungan fitur dengan damage grade

In [ ]:
order = sorted(train[config.TARGET].unique())
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col in zip(axes, ["foundation_type", "superstructure"]):
    ct = pd.crosstab(train[col], train[config.TARGET], normalize="index")[order]
    ct.plot(kind="barh", stacked=True, ax=ax, colormap="RdYlGn_r", width=0.8)
    ax.set_title(f"Damage grade per {col}")
    ax.set_xlabel("proporsi")
    ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "damage_by_structure.png", dpi=120)
plt.show()

In [ ]:
ct = pd.crosstab(train["district_id"], train[config.TARGET], normalize="index")[order]
ct.plot(kind="bar", stacked=True, figsize=(9, 4), colormap="RdYlGn_r", width=0.85)
plt.title("Damage grade per district_id")
plt.ylabel("proporsi")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "damage_by_district.png", dpi=120)
plt.show()

## Kesimpulan EDA
- Lokasi (`vdcmun_id`, `district_id`) dan jenis struktur/fondasi sangat berpengaruh pada tingkat kerusakan.
- `age_building = 999` perlu ditangani khusus.
- `ward_id` punya 945 level → diperlakukan numerik; `district_id` & `vdcmun_id` dikategorikan.
- Tidak ada missing value maupun ID duplikat.